#### User Interaction for Handwriting Recognition

##### Step 1: Targeted Extraction of Postcard Versos
###### Instead of typical data cleaning, we implemented a Targeted Inclusion Logic, filtering for high-density layouts with low AI confidence to specifically harvest the 'hardest' cases—where human intervention adds the most value.

##### Handwriting Selection Logic: Calibrated the EasyOCR engine to flag images with diverse text layouts (multiple bounding boxes) and a minimum aggregate character length (> 25 characters) but low average confidence scores (< 0.65). This logic identifies cards with dense, overlapping, or faded handwriting.
##### Manual Quality Control: Implemented a final human-verification step to filter out "false positives"—cards that are blank or only contain minimal printed text but were flagged by AI due to low-contrast ink.

In [ ]:
!pip install --user easyocr

In [7]:
import os
import sys

SCRATCH_ENV = "/scratch/leuven/387/vsc38790/my_python_env"
os.makedirs(SCRATCH_ENV, exist_ok=True)

print(f" EasyOCR installed in {SCRATCH_ENV}")

!{sys.executable} -m pip install --target={SCRATCH_ENV} --no-cache-dir easyocr

🚀 开始将 EasyOCR 安装到大容量目录: /scratch/leuven/387/vsc38790/my_python_env
这次绝对不会报硬盘配额不足了，请耐心等待安装完毕...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 65.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 51.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 60.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 72.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 76.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 79.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 78.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 141.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 107.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━

In [7]:
import sys
import os
import pandas as pd
import numpy as np
from PIL import Image
import warnings

warnings.filterwarnings('ignore')

# 1. HPC setting
SCRATCH_ENV = "/scratch/leuven/387/vsc38790/my_python_env"
sys.path.insert(0, SCRATCH_ENV)

import torch
torch.set_num_threads(1)

import easyocr

EXCEL_PATH = "/data/leuven/387/vsc38790/20230301_Postcards_Cleaned.xlsx"
IMAGE_ROOT_DIR = "/scratch/leuven/387/vsc38790/Dataset_0003_PicturePostcards"
OUTPUT_CSV = "challenge_backs_for_users.csv"
SAMPLE_SIZE = 100 
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.jp2')
MAX_IMAGE_SIDE = 1000 

# 2. Metadata Filtering
df = pd.read_excel(EXCEL_PATH)
df_filtered = df[df['Language'].str.contains('fre|dut|eng', na=False, case=False)]
df_filtered = df_filtered[df_filtered['Main title'].str.len() < 30]

print(f"Metadata filtering complete. Found {len(df_filtered)} matching records.")

# 3. Sampling
sample_size = min(SAMPLE_SIZE, len(df_filtered))
df_sample = df_filtered.sample(n=sample_size, random_state=42)

# 4. Initialize EasyOCR
reader = easyocr.Reader(['fr', 'nl', 'en'], gpu=False, model_storage_directory=SCRATCH_ENV)
challenge_cards = []

print(f"Running OCR on {sample_size} sampled images...\n")

for index, row in df_sample.iterrows():
    mms_id = str(row['MMS ID'])
    title = str(row['Main title'])
    resolver_url = str(row['Resolver URL'])
    language = str(row['Language'])
    
    ie_folder = resolver_url.split('/')[-2] if 'IE' in resolver_url else ""
    if not ie_folder: continue
        
    target_dir = os.path.join(IMAGE_ROOT_DIR, ie_folder)
    img_path = None
    
    # finding the back of the postcards
    if os.path.exists(target_dir):
        all_valid_images = []
        for root, _, files in os.walk(target_dir):
            for file in files:
                if file.lower().endswith(VALID_EXTENSIONS):
                    all_valid_images.append(os.path.join(root, file))
        
        if len(all_valid_images) >= 2:
            all_valid_images.sort()
            img_path = all_valid_images[-1]
        else:
            continue
                
    if not img_path: continue 
        
    # pic pre-processing
    try:
        pil_img = Image.open(img_path).convert('RGB')
        w, h = pil_img.size
        if max(w, h) > MAX_IMAGE_SIDE:
            scale = MAX_IMAGE_SIDE / max(w, h)
            pil_img = pil_img.resize((int(w*scale), int(h*scale)), Image.Resampling.LANCZOS)
        img_array = np.array(pil_img)
    except Exception: continue
        
    # OCR recognition
    try:
        result = reader.readtext(img_array)
    except Exception: continue
        
    num_boxes = len(result)
    if num_boxes == 0: continue
        
    confidences = [res[2] for res in result]
    avg_conf = np.mean(confidences)
    detected_texts = [res[1] for res in result]
    
    # count the total characters
    total_chars = sum(len(text) for text in detected_texts)
    
    if num_boxes >= 3 and avg_conf < 0.65 and total_chars > 25:
        print(f" MMS ID: {mms_id} | num_boxes: {num_boxes} | total_chars: {total_chars} | avg_conf: {avg_conf:.2f}")
        challenge_cards.append({
            'MMS_ID': mms_id,
            'Language': language,
            'Image_Path': img_path,
            'Ground_Truth': title,
            'AI_Confidence': round(avg_conf, 2),
            'AI_Detected_Text': " | ".join(detected_texts)
        })

# 5. Export Results
final_df = pd.DataFrame(challenge_cards)
final_df.to_csv(OUTPUT_CSV, index=False)

print(f"Exported {len(challenge_cards)} high-quality challenges to {OUTPUT_CSV}")

Using CPU. Note: This module is much faster with a GPU.


Metadata filtering complete. Found 7206 matching records.
Running OCR on 100 sampled images...

 MMS ID: 9992829509501488 | num_boxes: 40 | total_chars: 247 | avg_conf: 0.51
 MMS ID: 9992463506201488 | num_boxes: 9 | total_chars: 44 | avg_conf: 0.57
 MMS ID: 9992301905101488 | num_boxes: 23 | total_chars: 204 | avg_conf: 0.43
 MMS ID: 9992421301601488 | num_boxes: 13 | total_chars: 78 | avg_conf: 0.36
 MMS ID: 9992141466401488 | num_boxes: 21 | total_chars: 193 | avg_conf: 0.57
 MMS ID: 9992483892801488 | num_boxes: 21 | total_chars: 85 | avg_conf: 0.54
 MMS ID: 9992585382001488 | num_boxes: 17 | total_chars: 121 | avg_conf: 0.36
 MMS ID: 9992521503401488 | num_boxes: 14 | total_chars: 109 | avg_conf: 0.55
 MMS ID: 9992488191901488 | num_boxes: 7 | total_chars: 41 | avg_conf: 0.47
 MMS ID: 9992312710101488 | num_boxes: 24 | total_chars: 127 | avg_conf: 0.48
 MMS ID: 9992337806201488 | num_boxes: 11 | total_chars: 101 | avg_conf: 0.43
 MMS ID: 9992231133201488 | num_boxes: 29 | total_ch

In [9]:
import os
import shutil
import pandas as pd

csv_path = "/data/leuven/387/vsc38790/challenge_backs_for_users.csv"

output_dir = "/data/leuven/387/vsc38790/Final_Challenge_Images"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)
count = 0

for index, row in df.iterrows():
    original_path = str(row['Image_Path'])
    mms_id = str(row['MMS_ID'])

    if pd.notna(original_path) and os.path.exists(original_path):
        ext = os.path.splitext(original_path)[1]
        new_filename = f"{mms_id}_back{ext}"
        new_path = os.path.join(output_dir, new_filename)
        
        try:
            shutil.copy2(original_path, new_path)
            print(f"copied: {new_filename}")
            count += 1
        except Exception as e:
            print(f" Copy {mms_id} failedd: {e}")

copied: 9992829509501488_back.jpg
copied: 9992463506201488_back.jpg
copied: 9992301905101488_back.jpg
copied: 9992421301601488_back.jpg
copied: 9992141466401488_back.jpg
copied: 9992483892801488_back.jpg
copied: 9992585382001488_back.jpg
copied: 9992521503401488_back.jpg
copied: 9992488191901488_back.jpg
copied: 9992312710101488_back.jpg
copied: 9992337806201488_back.jpg
copied: 9992231133201488_back.jpg
copied: 9992700401401488_back.jpg
copied: 9992490909201488_back.jpg
copied: 9992541289601488_back.jpg
copied: 9992501708301488_back.jpg
copied: 9992408485701488_back.jpg
copied: 9992258475801488_back.jpg
copied: 9992192585301488_back.jpg
copied: 9992155262101488_back.jpg
copied: 9992155212801488_back.jpg
copied: 9992343686301488_back.jpg
copied: 9992457986201488_back.jpg
copied: 9992367839101488_back.jpg
copied: 9992616694201488_back.jpg
copied: 9992231128801488_back.jpg
copied: 9992694902501488_back.jpg
copied: 9992312709601488_back.jpg
copied: 9991828450101488_back.jpg
copied: 999250

##### Among the 62 postcards seleted, 6 are blank.